# Network & IT Support Ticket Analyzer

Notebook สาธิต Regex & Cleansing, PyThaiNLP Tokenization, Normalization, Topic Identification และ Rule-based NER สำหรับข้อความแจ้งปัญหา IT/Network

In [ ]:
!pip -q install pandas 'pythainlp>=5.3.2,<6.0'

import urllib.request
from pathlib import Path

# ใช้ relative paths ใน Colab และดึงไฟล์จาก repository เฉพาะเมื่อยังไม่มีไฟล์
REPO_RAW_BASE = 'https://raw.githubusercontent.com/OomyKung/NLP-Sub-Exam/main'

def fetch_if_missing(path, url):
    path = Path(path)
    if not path.exists():
        path.parent.mkdir(parents=True, exist_ok=True)
        urllib.request.urlretrieve(url, path)

fetch_if_missing('nlp_utils.py', f'{REPO_RAW_BASE}/nlp_utils.py')
fetch_if_missing('data/test_data.csv', f'{REPO_RAW_BASE}/data/test_data.csv')

from nlp_utils import analyze_text
import pandas as pd

## วิเคราะห์ข้อความตัวอย่าง

In [ ]:
sample = (
    'เน็ตห้อง Lab 402 ใช้งานไม่ได้ตั้งแต่ 09:30 น. '
    'Cisco Router IP 192.168.1.1 ping ไม่เจอ ด่วนมากครับ ติดต่อ 081-234-5678'
)
result = analyze_text(sample)
print('Language:', result['language'])
print('Topic:', result['topic'])
print('Priority:', result['priority'])
print('Cleaned text:', result['cleaned_text'])
print('Tokens:', result['filtered_tokens'])
print('Entities:')
for entity in result['entities']:
    print(f"  {entity['text']} -> {entity['label']}")

## วิเคราะห์ Dataset และวัด Accuracy

In [ ]:
data = pd.read_csv('data/test_data.csv')
rows = []
for _, row in data.iterrows():
    result = analyze_text(str(row['text']))
    rows.append({
        **row.to_dict(),
        'predicted_topic': result['topic'],
        'predicted_priority': result['priority'],
        'language': result['language'],
        'extracted_entities': '; '.join(
            f"{item['text']} ({item['label']})" for item in result['entities']
        ),
    })

analysis_results = pd.DataFrame(rows)
topic_accuracy = (analysis_results['expected_topic'] == analysis_results['predicted_topic']).mean()
priority_accuracy = (analysis_results['expected_priority'] == analysis_results['predicted_priority']).mean()
print(f'Topic Accuracy: {topic_accuracy:.1%}')
print(f'Priority Accuracy: {priority_accuracy:.1%}')
analysis_results.head()

In [ ]:
errors = analysis_results[
    (analysis_results['expected_topic'] != analysis_results['predicted_topic'])
    | (analysis_results['expected_priority'] != analysis_results['predicted_priority'])
]
print('Prediction errors:', len(errors))
errors[['id', 'text', 'expected_topic', 'predicted_topic', 'expected_priority', 'predicted_priority']]

In [ ]:
analysis_results.to_csv('analysis_results.csv', index=False, encoding='utf-8-sig')
print('Exported analysis_results.csv')